# 04 - Baseline (naive) predictors

**Influenza Season Forecasting** - Notebook 4 of 5

**Purpose:** Establish the naive floor that any real model in `05` must beat. No tuned models, no
ARIMA/Prophet, no feature engineering beyond what each baseline needs. Targets are the **smoothed**
`peak_week` and `peak_ili_pct` from `02` (ground truth). Per CLAUDE.md (baselines before models),
this notebook stops for review before any commit.

**These are NOT results.** They are the bar `05` must clear.

## Evaluation design (must mirror 05 or the comparison is invalid)

- **Leave-one-season-out (LOSO).** For each held-out season, a baseline is computed using only the
  other seasons (cross-sectional baselines) or only the held-out season's own data through a
  decision week W (within-season baseline). No baseline uses information a real forecaster at week W
  would not have.
- **Excluded from BOTH training pool and evaluation:** the two pandemic seasons (2009-10, 2020-21)
  and the pandemic-adjacent 2008-09, consistent with 02. This leaves **19 evaluation seasons**, the
  same set 05 will use, so the floor is comparable.
- **Season-week axis (wrap fix).** Peak-week error is measured on a linear season-week index
  `sw(week) = week-39 if week>=40 else week+13` (wk40=1 ... wk39=52). This makes wk52 and wk1 one
  week apart, not 51. All peak-week MAE / within-1 figures are in these weeks.
- **Leakage in the within-season baseline.** The centered 3-week smoother used to define the target
  looks one week past the peak; that is fine for a retrospective target but would leak at a decision
  week W. So baseline C reads the **raw** observed ILI through W (leakage-free), and is scored
  against the smoothed truth. The W cutoff is applied strictly on the same `sw` axis.
- **Fragile labels.** Peak-week metrics are reported twice: over all 19 seasons, and excluding the
  `fragile_peak_week` seasons from 02 (week label ambiguous to ~1 week). Severity (peak_ili) metrics
  are reported once; the fragile flag is about week timing, not height.

## Setup and reconstruction of cleaned data (02's committed logic)

02 persists nothing, so we rebuild `weekly`, `season_table`, and the strain series from 02's
reviewed code, then assert the reconstruction matches (19 eval seasons, 8 fragile among them).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path("data/raw")
RESULTS_DIR = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)
EXCLUDED = {"2008-09", "2009-10", "2020-21"}   # pandemic + pandemic-adjacent
DECISION_WEEKS = [8, 12, 16]                    # season-week (sw) cutoffs for baseline C

def season_of(year, week):
    sy = year if week >= 40 else year - 1
    return f"{sy}-{str(sy + 1)[2:]}", sy

def sw(week):
    """Linear season-week index: wk40=1 ... wk52=13 ... wk1=14 ... wk39=52."""
    return week - 39 if week >= 40 else week + 13

def sw_to_week(s):
    return s + 39 if s <= 13 else s - 13

In [ ]:
# ---- ILINet weekly series ----
ili = pd.read_csv(DATA_DIR / "ILINet.csv", skiprows=1, na_values=["X"])
_i = ili.apply(lambda r: season_of(int(r["YEAR"]), int(r["WEEK"])), axis=1)
ili["season"] = [x[0] for x in _i]; ili["ssy"] = [x[1] for x in _i]
ili["order"] = ili["WEEK"].apply(lambda w: w if w >= 40 else w + 100)
ili = ili.sort_values(["ssy", "order"]).reset_index(drop=True)
ili["sw"] = ili["WEEK"].apply(sw)

def _complete(g):
    sy = int(g["ssy"].iloc[0]); sp = sorted(g.loc[g["YEAR"] == sy, "WEEK"]); ep = sorted(g.loc[g["YEAR"] == sy + 1, "WEEK"])
    return bool(sp and sp[0] == 40 and ep and ep[0] == 1 and ep[-1] == 39)
complete = [s for s, g in ili.groupby("season") if _complete(g)]
weekly = ili[ili["season"].isin(complete)].copy()

# ---- season_table: smoothed targets + fragile flag ----
rows = []
for s, g in weekly.groupby("season"):
    g = g.sort_values("order")
    sm3 = g["% WEIGHTED ILI"].rolling(3, center=True).mean()
    sm5 = g["% WEIGHTED ILI"].rolling(5, center=True).mean()
    rows.append(dict(season=s, ssy=int(g["ssy"].iloc[0]),
                     peak_week=int(g.loc[sm3.idxmax(), "WEEK"]), peak_ili_pct=round(float(sm3.max()), 3),
                     peak_week_sm5=int(g.loc[sm5.idxmax(), "WEEK"])))
season_table = pd.DataFrame(rows).sort_values("ssy").reset_index(drop=True)
season_table["fragile_peak_week"] = season_table["peak_week"] != season_table["peak_week_sm5"]
season_table["sw_true"] = season_table["peak_week"].apply(sw)

# ---- dominant strain stitch (for baseline D) ----
def load_nrevss(f):
    d = pd.read_csv(DATA_DIR / f, skiprows=1, na_values=["X", "XX"])
    ii = d.apply(lambda r: season_of(int(r["YEAR"]), int(r["WEEK"])), axis=1)
    d["season"] = [x[0] for x in ii]; d["ssy"] = [x[1] for x in ii]; return d
def dominant(d):
    col = lambda n: d[n] if n in d.columns else 0
    bk = pd.DataFrame({"season": d["season"], "ssy": d["ssy"], "A(H1N1)": col("A (H1)") + col("A (2009 H1N1)"),
                       "A(H3N2)": col("A (H3)"), "B": col("B") + col("BVic") + col("BYam")})
    a = bk.groupby(["season", "ssy"])[["A(H1N1)", "A(H3N2)", "B"]].sum().reset_index()
    a["dominant_strain"] = a[["A(H1N1)", "A(H3N2)", "B"]].idxmax(axis=1); return a
cb = dominant(load_nrevss("ICL_NREVSS_Combined_prior_to_2015_16.csv"))
ph = dominant(load_nrevss("ICL_NREVSS_Public_Health_Labs.csv"))
strain = pd.concat([cb[cb["ssy"] <= 2014], ph[ph["ssy"] >= 2015]])[["season", "dominant_strain"]]
season_table = season_table.merge(strain, on="season", how="left")

# ---- evaluation set ----
EVAL = [s for s in season_table["season"] if s not in EXCLUDED]
ev = season_table[season_table["season"].isin(EVAL)].reset_index(drop=True)
assert len(EVAL) == 19
assert int(ev["fragile_peak_week"].sum()) == 8
print("reconstruction OK. eval seasons:", len(EVAL))
print("excluded:", sorted(EXCLUDED))
print("fragile in eval (n=%d):" % ev["fragile_peak_week"].sum(), ev.loc[ev["fragile_peak_week"], "season"].tolist())
print("strain counts in eval:", ev["dominant_strain"].value_counts().to_dict())

## Metric helpers

`peak_week`: mean absolute error in season-weeks, and % of seasons within +/-1 week.
`peak_ili_pct`: MAE and RMSE. All under LOSO.

In [ ]:
def wk_metrics(df, pred_sw):
    err = (df[pred_sw] - df["sw_true"]).abs()
    return round(float(err.mean()), 2), round(float(100 * (err <= 1).mean()), 1)

def ili_metrics(df, pred, truth="peak_ili_pct"):
    e = df[pred] - df[truth]
    return round(float(e.abs().mean()), 3), round(float(np.sqrt((e ** 2).mean())), 3)

def record(name, df, has_week, pred_sw=None, pred_ili=None, n=None):
    r = {"baseline": name, "eval_n": int(n if n is not None else len(df))}
    if has_week:
        mae_a, w1_a = wk_metrics(df, pred_sw)
        mae_f, w1_f = wk_metrics(df[~df["fragile_peak_week"]], pred_sw)
        r.update(pw_MAE_all=mae_a, pw_within1_all=w1_a, pw_MAE_exfrag=mae_f, pw_within1_exfrag=w1_f)
    else:
        r.update(pw_MAE_all=np.nan, pw_within1_all=np.nan, pw_MAE_exfrag=np.nan, pw_within1_exfrag=np.nan)
    if pred_ili is not None:
        mae, rmse = ili_metrics(df, pred_ili)
        r.update(ili_MAE=mae, ili_RMSE=rmse)
    else:
        r.update(ili_MAE=np.nan, ili_RMSE=np.nan)
    return r

summary = []

## Baseline A - Climatology (zero-information floor)

LOSO: predict `peak_week` = median of the other seasons' peak weeks (on the sw axis), and
`peak_ili_pct` = mean of the other seasons' peak severities. Uses no current-season information.

In [ ]:
A = ev.copy()
A["pw_pred_sw"] = [int(round(ev.loc[ev["season"] != s, "sw_true"].median())) for s in A["season"]]
A["ili_pred"] = [float(ev.loc[ev["season"] != s, "peak_ili_pct"].mean()) for s in A["season"]]
summary.append(record("A_climatology", A, True, "pw_pred_sw", "ili_pred"))
print("A climatological peak_week guess (sw):", sorted(A["pw_pred_sw"].unique()),
      "-> wk", sorted({sw_to_week(x) for x in A["pw_pred_sw"]}))
print("A:", summary[-1])

## Baseline B - Persistence (prior season)

Predict the held-out season's peak from the **immediately preceding** included season.
Edge case: if the preceding season is one of the excluded (pandemic / pandemic-adjacent) seasons,
no valid one-season persistence exists, so that season is dropped from B's evaluation (rather than
silently reaching back several years). Dropped: the first season (no prior), and the two seasons
whose predecessor is an excluded pandemic season.

In [ ]:
chron = season_table.sort_values("ssy")["season"].tolist()
prior = {}
for s in EVAL:
    j = chron.index(s); p = chron[j - 1] if j > 0 else None
    prior[s] = p if (p in EVAL) else None
B_seasons = [s for s in EVAL if prior[s]]
B = ev[ev["season"].isin(B_seasons)].copy()
B["pw_pred_sw"] = [int(season_table.loc[season_table["season"] == prior[s], "sw_true"].iloc[0]) for s in B["season"]]
B["ili_pred"] = [float(season_table.loc[season_table["season"] == prior[s], "peak_ili_pct"].iloc[0]) for s in B["season"]]
summary.append(record("B_persistence", B, True, "pw_pred_sw", "ili_pred"))
print("B persistence eval n =", len(B), "| dropped:", [s for s in EVAL if not prior[s]])
print("B:", summary[-1])

## Baseline C - Within-season running max at decision week W

Standing at season-week W of the held-out season, predict `peak_ili_pct` = max **raw** ILI observed
through W and `peak_week` = the week of that running max. Raw (not smoothed) to avoid the centered
smoother peeking past W. Computed at W = 8, 12, 16 (through wk47, wk51, wk3) to show how the naive
within-season guess sharpens as more of the season is seen. No data after W is used.

**Read C's error carefully.** C compares the **raw** observed ILI through W against the **smoothed**
target. So part of C's error is residual holiday / denominator noise that was deliberately smoothed
out of the truth, not pure forecasting error. This is the right setup (a real forecaster at week W
has only raw reports, not a retrospective smoother), but it means C's gap to the target overstates
how much a like-for-like forecaster would miss by.

In [ ]:
for W in DECISION_WEEKS:
    C = ev.copy(); pw_sw, ili_pred = [], []
    for s in C["season"]:
        g = weekly[(weekly["season"] == s) & (weekly["sw"] <= W)]
        mx = g["% WEIGHTED ILI"].max()
        mw = int(g.loc[g["% WEIGHTED ILI"] == mx, "WEEK"].iloc[0])
        pw_sw.append(sw(mw)); ili_pred.append(float(mx))
    C["pw_pred_sw"] = pw_sw; C["ili_pred"] = ili_pred
    summary.append(record(f"C_runmax_W{W}_thru_wk{sw_to_week(W)}", C, True, "pw_pred_sw", "ili_pred"))
    print(summary[-1])

## Baseline D - Climatology by dominant strain (exploratory)

LOSO: predict `peak_ili_pct` = mean severity among training seasons sharing the held-out season's
dominant strain (fallback to overall climatology if no same-strain training season). **Exploratory
only:** per-strain n is tiny (H3N2=10, H1N1=6, B=3 across the eval set), so this is a hypothesis
probe, not a usable predictor. Severity only; strain gives no natural peak-week climatology.

In [ ]:
Dt = ev.copy(); dpred = []
for s in Dt["season"]:
    d = Dt.loc[Dt["season"] == s, "dominant_strain"].iloc[0]
    pool = ev[(ev["season"] != s) & (ev["dominant_strain"] == d)]
    dpred.append(float(pool["peak_ili_pct"].mean()) if len(pool) else float(ev.loc[ev["season"] != s, "peak_ili_pct"].mean()))
Dt["ili_pred"] = dpred
summary.append(record("D_climatology_by_strain (exploratory)", Dt, False, pred_ili="ili_pred"))
print("D:", summary[-1])

## Summary table and the floor on each target

Saved to `results/` as markdown and JSON (not CSV: `*.csv` is gitignored project-wide, so a CSV
here would be silently untracked). The cell also prints which baseline is the strongest floor for
each target. These are floors, not results.

In [ ]:
summary_df = pd.DataFrame(summary)[
    ["baseline", "eval_n", "pw_MAE_all", "pw_within1_all", "pw_MAE_exfrag", "pw_within1_exfrag", "ili_MAE", "ili_RMSE"]]
print(summary_df.to_string(index=False))

# save (md + json; avoid csv on purpose, tabulate-free markdown)
def to_md(df):
    cols = list(df.columns)
    head = "| " + " | ".join(cols) + " |"
    sep = "| " + " | ".join("---" for _ in cols) + " |"
    body = ["| " + " | ".join("" if pd.isna(v) else str(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join([head, sep] + body)
(RESULTS_DIR / "04_baseline_summary.md").write_text(
    "# 04 baseline summary (LOSO, 19 non-pandemic seasons)\n\n" + to_md(summary_df) + "\n", encoding="utf-8")
summary_df.to_json(RESULTS_DIR / "04_baseline_summary.json", orient="records", indent=2)
print("\nsaved results/04_baseline_summary.md and .json")

# strongest floor per target
wk_rows = summary_df.dropna(subset=["pw_MAE_all"])
best_pw_mae = wk_rows.loc[wk_rows["pw_MAE_all"].idxmin()]
best_pw_w1 = wk_rows.loc[wk_rows["pw_within1_all"].idxmax()]
best_ili = summary_df.loc[summary_df["ili_MAE"].idxmin()]
print("\nFloor for peak_week (lowest MAE):  {} -> MAE {} wk".format(best_pw_mae["baseline"], best_pw_mae["pw_MAE_all"]))
print("Floor for peak_week (highest +/-1): {} -> {}%".format(best_pw_w1["baseline"], best_pw_w1["pw_within1_all"]))
print("Floor for peak_ili_pct (lowest MAE): {} -> MAE {}".format(best_ili["baseline"], best_ili["ili_MAE"]))

**Read of the floors (descriptive, not final results).**

- **peak_week (timing) is hard for every naive baseline.** All cross-sectional baselines land around
  3.3-3.8 weeks MAE and put only ~26-37% of seasons within +/-1 week. Persistence has the lowest
  cross-sectional MAE; the within-season running max at W=16 (through early January) gives the
  highest +/-1 rate. The template's "+/-1 week on >=70% of seasons" sits far above this naive floor,
  so 05 has real room and real need to beat it.
- **peak_ili_pct (severity):** the climatology mean is the cross-sectional floor; persistence is
  notably worse (severity bounces season to season). The within-season running max sharpens with W
  (W=8 far worse than climatology, W=12 roughly matches it, W=16 beats it), which is the expected
  shape of a within-season floor. Strain-climatology (D) is worse than plain climatology, so dominant
  strain alone does not lower the severity floor at this sample size.
- Comparisons must be lead-time matched: a within-season baseline at W is only a fair floor for a
  model also standing at W.

## Preliminary finding: severity is forecastable, timing is near the noise floor

This is the headline of this notebook, stated as a **preliminary finding** (19 seasons, naive
baselines only), not a settled result.

**Peak-week timing has no good naive floor.** Every cross-sectional baseline sits at 3.3-3.8 weeks
MAE and puts at most ~37% of seasons within +/-1 week, and the within-season running max is *worse*
than climatology at short lead (W=8: 9.2 wk MAE) before improving. On 19 seasons, peak-week timing
looks close to the noise floor at these lead times. Two consequences:

- A `05` timing model that posts high peak-week accuracy warrants **suspicion of overfitting**, not
  celebration. With this floor, large honest gains are unlikely; a big jump more plausibly signals
  leakage or fitting to 19-season noise. Sanity-check any such result hard.
- The template's "+/-1 week on >=70% of seasons" target is **likely unreachable honestly** at these
  lead times on this sample. It should be treated as an aspirational goal, not a deliverable (this
  also feeds the open question for Dr. Mitra about whether those metrics are goals vs deliverables).

**Severity is a different story.** `peak_ili_pct` shows a clean within-season sharpening
(climatology MAE 1.34, improving to 0.84 once early January is observed), so height is forecastable
in a way timing is not. Frame the two targets as **asymmetric**: severity is tractable, timing is
hard. `05` should be expected to beat the severity floor and should be judged skeptically on timing.

## Next step

`05_forecasting.ipynb`: ARIMA / Prophet (and an optional severity classifier) through one LOSO
harness, scored on these same smoothed targets and the same 19 seasons, reported against these
floors with interval calibration alongside point error.